In [2]:
import h5py
import pandas as pd 
import xarray as xr
import os
import cfgrib
from pathlib import Path

def h5_to_df(file_path):
    ds = xr.open_dataset(file_path)
    df = ds.to_dataframe().reset_index()
    return df

In [4]:
Stations_df = pd.read_csv("../config/Station_Locations.csv").reset_index()
Stations_df.info()
Valid_Station_df = pd.read_csv("../data/processed/Valid_Stations.csv").reset_index()
Valid_Station_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 505 entries, 0 to 504
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   index            505 non-null    int64  
 1   Station_id       505 non-null    int64  
 2   Station_Name     505 non-null    str    
 3   Latitude_Start   505 non-null    float64
 4   Latitude_End     505 non-null    float64
 5   Longitude_Start  505 non-null    float64
 6   Longitude_End    505 non-null    float64
 7   latitude         499 non-null    float64
 8   longitude        499 non-null    float64
dtypes: float64(6), int64(2), str(1)
memory usage: 42.1 KB
<class 'pandas.DataFrame'>
RangeIndex: 399 entries, 0 to 398
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   level_0          399 non-null    int64  
 1   index            399 non-null    int64  
 2   Station_id       399 non-null    int64  
 3   Station_Name  

In [ ]:
import tqdm


folder = "../data/raw/3RIMG_L2G_AOD"

rows = []
for root, dirs, files in tqdm.tqdm(os.walk(folder)):
    for file in files:
        full_path = os.path.join(root, file)
        ds = xr.open_dataset(full_path)

        for station in Valid_Station_df.itertuples():
            lat = slice(station.Latitude_Start, station.Latitude_End)
            lon = slice(station.Longitude_Start, station.Longitude_End)

           
            point = ds.sel(latitude=lat, longitude=lon)
            mean_ds = point.mean(dim=['latitude', 'longitude'])

            df = mean_ds.to_dataframe().reset_index()

            df["Station_Id"] = station.Station_id  
            rows.append(df) 

master_AOD = pd.concat(rows, ignore_index=True)

master_AOD  = master_AOD.dropna()

157it [07:29,  2.94s/it]

In [1]:
master_AOD_DailyAvg = master_AOD.copy()
master_AOD_DailyAvg["time"] = pd.to_datetime(master_AOD_DailyAvg["time"]).dt.date
master_AOD_DailyAvg = master_AOD_DailyAvg.groupby(["Station_Id", "time"]).mean().reset_index()

master_AOD_DailyAvg.head(20)
master_AOD_DailyAvg.to_csv("../data/processed/master_AOD.csv", index=False)

NameError: name 'master_AOD' is not defined

In [ ]:
# Folder containing GRIB files
folder = Path("../data/raw/era5")

# # Get all grib files
files = list(folder.glob("*.grib"))

all_dfs = []

for file in files:
    try:
        print(f"Opening {file.name}")

        # Open GRIB file
        grib_datasets = cfgrib.open_datasets(file)

        # Convert to dataframe While iterating through all the station locations and slicing the dataset accordingly
        for ds in grib_datasets:
            for station in Valid_Station_df.itertuples():
                lat = slice(station.Latitude_start, station.Latitude_End)
                lon = slice(station.Longitude_Start, station.Longitude_End)

                df = ds.sel(latitude=lat, longitude=lon).to_dataframe().reset_index()
                df["Station_Id"] = station.Station_id

                all_dfs.append(df)

    except Exception as e:
        print(f"Failed on {file.name}: {e}")

# Merge everything
era5_df = pd.concat(all_dfs, ignore_index=True)

era5_df.drop(columns=["latitude", "longitude", "number", "step", "surface", "valid_time"], inplace=True)

print(era5_df.head())
print(era5_df.columns)

Opening era5_2025_2.grib


Ignoring index file '../data/raw/era5/era5_2025_2.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_2.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_2.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_2.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_2.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_2.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_2.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_2.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_2.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_2.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_2.grib.5b7b6

Opening era5_2025_1.grib


Ignoring index file '../data/raw/era5/era5_2025_1.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_1.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_1.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_1.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_1.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_1.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_1.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_1.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_1.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../data/raw/era5/era5_2025_1.grib.5b7b6.idx' incompatible with GRIB file
/home/victus7/Desktop/Personal-Project/AirPollution/.venv/li

                 time            sp         blh       tcc       u10       v10  \
0 2025-06-01 00:00:00  92801.320312  515.269043  0.991119  2.733932  1.606140   
1 2025-06-01 01:00:00  92826.812500  495.264954  1.000000  2.588852  1.731461   
2 2025-06-01 02:00:00  92899.750000  521.940674  0.966858  3.018280  2.250443   
3 2025-06-01 03:00:00  92918.281250  509.364563  0.878601  3.292694  2.654816   
4 2025-06-01 04:00:00  92981.468750  612.487427  0.795227  3.466675  2.631042   

          t2m         d2m         skt  Station_Id  ssrd   e  tp  
0  297.990967  296.048828  295.571960           1   NaN NaN NaN  
1  298.046143  296.177002  295.769775           1   NaN NaN NaN  
2  298.303955  296.364716  296.822510           1   NaN NaN NaN  
3  298.304199  296.366455  297.883545           1   NaN NaN NaN  
4  298.482178  296.313721  300.120117           1   NaN NaN NaN  
Index(['time', 'sp', 'blh', 'tcc', 'u10', 'v10', 't2m', 'd2m', 'skt',
       'Station_Id', 'ssrd', 'e', 'tp'],
      

In [15]:

era5_DailyAvg = era5_df.copy()
era5_DailyAvg["time"] = pd.to_datetime(era5_DailyAvg["time"]).dt.date
era5_DailyAvg = era5_DailyAvg.groupby(["Station_Id", "time"]).mean().reset_index()
era5_DailyAvg.describe()
era5_DailyAvg.to_csv("../data/processed/era5_data.csv", index=False)

The below code extracts the data and also writes to the valid stations available 

In [5]:
def Cleaner_Extractor(file_path):
    cpcb = pd.read_csv(file_path).reset_index()
    cpcb = cpcb.dropna(subset=["PM2.5 (µg/m³)"])  
    cpcb = cpcb.rename(columns={"PM2.5 (µg/m³)": "PM2.5",
                                "index": "time"})
    cpcb = cpcb.rename(columns={"time": "Index",
                                "Timestamp": "time"})
    return cpcb[["time", "PM2.5"]]

folder_path = Path("../data/raw/PM_CPCB/")

all_CPCB_dfs = []
skipped = 0
Valid_Station = []

# Loop through all files ending in .csv
for file in folder_path.glob("*.csv"):
    print(f"Processing: {file.name}")
    df = Cleaner_Extractor(file)
    
    #extract station name from file name and add as a column
    station_name = file.stem.split("_")[1]  # Assuming the station name is the second part of the file name before an underscore
    try : 
        Station = Stations_df[Stations_df['Station_Name'] == station_name]
        Station_id = Station['Station_id'].values[0]
        Valid_Station.append(Station.head(1))
    except IndexError:
        print(f"Station ID not found for {station_name}")
        skipped += 1 
        continue
    df["Station_Id"] = Station_id


    df = df.dropna().reset_index()

    df["time"] = pd.to_datetime(df["time"]).dt.date # the exact time is of no use, only the dat as the project focuses on the daily average of PM2.5
    # print(df.describe())

    df = df.groupby("time").mean().reset_index()

    #feature engineering 
    # df["pm25_lag_1"] = df["PM2.5"].shift(1)
    # df["pm25_lag_3"] = df["PM2.5"].shift(3)
    # df["pm25_lag_7"] = df["PM2.5"].shift(7)

    df = df.dropna().reset_index(drop=True)  # Drop rows with NaN values after creating lag features
    print(df.head())

    all_CPCB_dfs.append(df)

master_CPCB = pd.concat(all_CPCB_dfs, ignore_index=True)
# master_CPCB.to_csv("../data/processed/cpcb_data.csv", index=False)    
print(f"Total files processed: {len(all_CPCB_dfs)}")
print(f"Total files skipped due to missing Station ID: {skipped}")
Valid_Station_df = pd.concat(Valid_Station).to_csv("../data/processed/Valid_Stations.csv", index=False)


Processing: CPCB_LumpyngngadShillong_2025.csv
Station ID not found for LumpyngngadShillong
Processing: CPCB_DeopurDhule_2025.csv
         time       index      PM2.5  Station_Id
0  2025-01-01   47.500000  42.861250       371.0
1  2025-01-02  143.500000  37.275104       371.0
2  2025-01-03  220.921569  37.151373       371.0
3  2025-01-05  451.000000  35.790351       371.0
4  2025-01-06  527.500000  42.571979       371.0
Processing: CPCB_RIICOInd._2025.csv
         time       index       PM2.5  Station_Id
0  2025-01-01   45.956522   54.145000       441.0
1  2025-01-02  137.913043   68.738261       441.0
2  2025-01-03  238.954023  122.247241       441.0
3  2025-01-04  327.245902  112.489344       441.0
4  2025-01-05  439.776316   78.389474       441.0
Processing: CPCB_H.B.Colony_2025.csv
         time        index      PM2.5  Station_Id
0  2025-01-01    48.609195  92.829885       494.0
1  2025-01-02   140.000000  31.200000       494.0
2  2025-01-09   807.000000  99.000000       494.0
3  2